In [20]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import time

In [21]:
data_raw = pd.read_csv("../data/input_data/data_imputed.csv")
print(data_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 133887 entries, 0 to 133886
Data columns (total 16 columns):
 #   Column                           Non-Null Count   Dtype
---  ------                           --------------   -----
 0   ID_VICTIMA                       133887 non-null  str  
 1   ORIGEN_REPORTE                   133887 non-null  str  
 2   FECHA_NACIMIENTO                 133887 non-null  str  
 3   SEXO                             133887 non-null  str  
 4   FECHA_DESAPARICION               133887 non-null  str  
 5   FECHA_REGISTRO                   133887 non-null  str  
 6   ESTATUS_VICTIMA                  133887 non-null  str  
 7   CVE_ENT                          133887 non-null  int64
 8   ENTIDAD                          133887 non-null  str  
 9   CVE_MUN                          133887 non-null  int64
 10  MUNICIPIO                        133887 non-null  str  
 11  SEXO_MAP                         133887 non-null  int64
 12  ESTATUS_MAP                      133887 n

## Categoría de Edad para el rango

In [22]:
# EDAD = FECHA_DESAPARICION - FECHA_NACIMIENTO
data_raw["EDAD"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce') - pd.to_datetime(data_raw["FECHA_NACIMIENTO"], errors='coerce')
data_raw["EDAD"] = data_raw["EDAD"].dt.days // 365

data_raw["EDAD"] = pd.cut(
    data_raw["EDAD"],
    bins=[-1, 11, 17, 29, 59, 200],
    labels=["Niño", "Adolescente", "Joven", "Adulto", "Adulto Mayor"],
    include_lowest=True
)

print(data_raw["EDAD"].value_counts())

EDAD
Adulto          66784
Joven           55622
Adolescente      5068
Adulto Mayor     3034
Niño             2672
Name: count, dtype: int64


In [23]:
# Extraer campos de la Fecha de Desaparicion para análisis temporal
data_raw["AÑO_DESAPARICION"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce').dt.year
data_raw["MES_DESAPARICION"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce').dt.month
data_raw["DIA_SEMANA_DESAPARICION"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce').dt.dayofweek

## Categoría de Tiempo de Reporte

In [24]:
# Restar la fecha de desaparicion con la de reporte para saber si fue inmediato o no, esto para un análisis posterior y ver si hay alguna relación entre el tiempo de reporte y otras variables
data_raw["TIEMPO_REPORTE"] = pd.to_datetime(data_raw["FECHA_REGISTRO"], errors='coerce') - pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce')
data_raw["TIEMPO_REPORTE"] = data_raw["TIEMPO_REPORTE"].dt.days

data_raw["TIEMPO_REPORTE"] = pd.cut(
    data_raw["TIEMPO_REPORTE"],
    bins=[-np.inf, 1, 7, 30, np.inf],
    labels=["Inmediato", "Rapido", "Tardio", "Muy Tardio"],
    include_lowest=True
)

print(data_raw["TIEMPO_REPORTE"].value_counts())

TIEMPO_REPORTE
Inmediato     128522
Muy Tardio      2201
Rapido          2127
Tardio          1037
Name: count, dtype: int64


## Transacciones

In [25]:
dfFinal = data_raw[['EDAD', 'SEXO', 'AÑO_DESAPARICION', 'MES_DESAPARICION', 'TIEMPO_REPORTE', 
                    'ESTATUS_VICTIMA', 'ENTIDAD']]

transacciones = []

for _, row in dfFinal.iterrows():
    trans = [
        f"EDAD_{row['EDAD']}",
        f"SEXO_{row['SEXO']}",
        f"AÑO_{row['AÑO_DESAPARICION']}",
        f"MES_{row['MES_DESAPARICION']}",
        f"TIEMPO_{row['TIEMPO_REPORTE']}",
        f"ESTATUS_{row['ESTATUS_VICTIMA']}",
        f"ENTIDAD_{row['ENTIDAD']}"
    ]
    transacciones.append(trans)

print("Primeras 5 transacciones:")
print(transacciones[:5])

Primeras 5 transacciones:
[['EDAD_Joven', 'SEXO_CONFIDENCIAL', 'AÑO_2012', 'MES_8', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_DURANGO'], ['EDAD_Adulto', 'SEXO_HOMBRE', 'AÑO_2025', 'MES_9', 'TIEMPO_Inmediato', 'ESTATUS_DESAPARECIDA', 'ENTIDAD_SINALOA'], ['EDAD_Adulto', 'SEXO_CONFIDENCIAL', 'AÑO_2020', 'MES_1', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_CIUDAD DE MÉXICO'], ['EDAD_Adulto', 'SEXO_CONFIDENCIAL', 'AÑO_2023', 'MES_2', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_BAJA CALIFORNIA'], ['EDAD_Adulto', 'SEXO_CONFIDENCIAL', 'AÑO_2023', 'MES_2', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_BAJA CALIFORNIA']]


## Codificación de Transacciones

In [26]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transacciones).transform(transacciones)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print("Primeras 5 filas de la matriz codificada:")
print(df_encoded.head())

Primeras 5 filas de la matriz codificada:
   AÑO_1961  AÑO_1962  AÑO_1964  AÑO_1967  AÑO_1968  AÑO_1969  AÑO_1970  \
0     False     False     False     False     False     False     False   
1     False     False     False     False     False     False     False   
2     False     False     False     False     False     False     False   
3     False     False     False     False     False     False     False   
4     False     False     False     False     False     False     False   

   AÑO_1971  AÑO_1972  AÑO_1973  ...  MES_8  MES_9  SEXO_CONFIDENCIAL  \
0     False     False     False  ...   True  False               True   
1     False     False     False  ...  False   True              False   
2     False     False     False  ...  False  False               True   
3     False     False     False  ...  False  False               True   
4     False     False     False  ...  False  False               True   

   SEXO_HOMBRE  SEXO_INDETERMINADO  SEXO_MUJER  TIEMPO_Inmediato  \


## Construcción de FP-Tree y extracción de Itemsets Frecuentes con FP-Growth

In [27]:
inicio = time.time()
rasgosFrecuentes_fpgrowth = fpgrowth(df_encoded, min_support=0.1, use_colnames=True)
fin = time.time()

print(f"Rasgos Frecuentes (FP-Growth): {rasgosFrecuentes_fpgrowth.shape[0]}")
print(f"Tiempo de ejecución de FP-Growth: {fin - inicio:.4f} segundos")
print("\nPrimeros 10 itemsets frecuentes:")
print(rasgosFrecuentes_fpgrowth.head(10))

Rasgos Frecuentes (FP-Growth): 66
Tiempo de ejecución de FP-Growth: 1.1505 segundos

Primeros 10 itemsets frecuentes:
    support                           itemsets
0  0.959929      frozenset({TIEMPO_Inmediato})
1  0.415440            frozenset({EDAD_Joven})
2  0.367093  frozenset({ESTATUS_CONFIDENCIAL})
3  0.367093     frozenset({SEXO_CONFIDENCIAL})
4  0.596630  frozenset({ESTATUS_DESAPARECIDA})
5  0.498809           frozenset({EDAD_Adulto})
6  0.482989           frozenset({SEXO_HOMBRE})
7  0.192319                 frozenset({MES_1})
8  0.147259            frozenset({SEXO_MUJER})
9  0.116942              frozenset({AÑO_2021})


## Comparación con Apriori

In [28]:
from mlxtend.frequent_patterns import apriori

inicio = time.time()
rasgosFrecuentes_apriori = apriori(df_encoded, min_support=0.1, use_colnames=True)
fin = time.time()

print(f"Rasgos Frecuentes (Apriori): {rasgosFrecuentes_apriori.shape[0]}")
print(f"Tiempo de ejecución de Apriori: {fin - inicio:.4f} segundos")
print(f"\nDiferencia de itemsets: {rasgosFrecuentes_fpgrowth.shape[0] - rasgosFrecuentes_apriori.shape[0]}")

Rasgos Frecuentes (Apriori): 66
Tiempo de ejecución de Apriori: 0.0607 segundos

Diferencia de itemsets: 0


## Reglas de Asociación

In [30]:
rules = association_rules(
    rasgosFrecuentes_fpgrowth,
    metric="confidence",
    min_threshold=0.5
)

rules = rules[(rules['confidence'] >= 0.5) & (rules['lift'] > 1)]

rules = rules.sort_values(by='lift', ascending=False)

print(f"Reglas de Asociación (Total): {rules.shape[0]}")
print("\nPrimeras 10 reglas:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Reglas de Asociación (Total): 90

Primeras 10 reglas:
                                          antecedents  \
19                  frozenset({ESTATUS_CONFIDENCIAL})   
20                     frozenset({SEXO_CONFIDENCIAL})   
23  frozenset({TIEMPO_Inmediato, ESTATUS_CONFIDENC...   
24   frozenset({TIEMPO_Inmediato, SEXO_CONFIDENCIAL})   
26                  frozenset({ESTATUS_CONFIDENCIAL})   
27                     frozenset({SEXO_CONFIDENCIAL})   
30         frozenset({EDAD_Joven, SEXO_CONFIDENCIAL})   
29      frozenset({ESTATUS_CONFIDENCIAL, EDAD_Joven})   
90  frozenset({TIEMPO_Inmediato, MES_1, SEXO_CONFI...   
89  frozenset({TIEMPO_Inmediato, ESTATUS_CONFIDENC...   

                                          consequents   support  confidence  \
19                     frozenset({SEXO_CONFIDENCIAL})  0.367093         1.0   
20                  frozenset({ESTATUS_CONFIDENCIAL})  0.367093         1.0   
23                     frozenset({SEXO_CONFIDENCIAL})  0.367093         1.0   
24

## Resultados de Reglas

In [ ]:
for i, row in rules.iterrows():
    print(f"{set(row['antecedents'])} → {set(row['consequents'])}")
    print(f"support: {row['support']:.3f}, confidence: {row['confidence']:.3f}, lift: {row['lift']:.3f}")
    print("-----")

{'ESTATUS_CONFIDENCIAL'} → {'SEXO_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'SEXO_CONFIDENCIAL'} → {'ESTATUS_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL'} → {'SEXO_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'TIEMPO_Inmediato', 'SEXO_CONFIDENCIAL'} → {'ESTATUS_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'ESTATUS_CONFIDENCIAL'} → {'TIEMPO_Inmediato', 'SEXO_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'SEXO_CONFIDENCIAL'} → {'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'EDAD_Joven', 'SEXO_CONFIDENCIAL'} → {'ESTATUS_CONFIDENCIAL'}
support: 0.153, confidence: 1.000, lift: 2.724
-----
{'ESTATUS_CONFIDENCIAL', 'EDAD_Joven'} → {'SEXO_CONFIDENCIAL'}
support: 0.153, confidence: 1.000, lift: 2.724
-----
{'TIEMPO_Inmediato', 'MES_1', 'SEXO_CONFIDENCIAL'} → {'ESTATUS_CONFIDENCIAL'

In [ ]:
# Filtrar reglas que no contengan la palabra 'CONFIDENCIAL'
filtered_rules = rules[
    ~rules['antecedents'].astype(str).str.contains("CONFIDENCIAL") &
    ~rules['consequents'].astype(str).str.contains("CONFIDENCIAL")
]

# Mostrar todas las reglas filtradas con su información detallada
for i, row in filtered_rules.iterrows():
    print(f"{set(row['antecedents'])} → {set(row['consequents'])}")
    print(f"support: {row['support']:.3f}, confidence: {row['confidence']:.3f}, lift: {row['lift']:.3f}")
    print("-----")

print("Total de reglas útiles:", len(filtered_rules))


{'ESTATUS_DESAPARECIDA', 'EDAD_Adulto'} → {'SEXO_HOMBRE'}
support: 0.216, confidence: 0.804, lift: 1.665
-----
{'ESTATUS_DESAPARECIDA', 'TIEMPO_Inmediato', 'EDAD_Adulto'} → {'SEXO_HOMBRE'}
support: 0.199, confidence: 0.803, lift: 1.662
-----
{'ESTATUS_DESAPARECIDA', 'EDAD_Adulto'} → {'SEXO_HOMBRE', 'TIEMPO_Inmediato'}
support: 0.199, confidence: 0.741, lift: 1.641
-----
{'ESTATUS_DESAPARECIDA', 'EDAD_Joven'} → {'SEXO_HOMBRE', 'TIEMPO_Inmediato'}
support: 0.181, confidence: 0.734, lift: 1.625
-----
{'SEXO_HOMBRE', 'EDAD_Joven'} → {'ESTATUS_DESAPARECIDA', 'TIEMPO_Inmediato'}
support: 0.181, confidence: 0.896, lift: 1.601
-----
{'SEXO_MUJER'} → {'ESTATUS_DESAPARECIDA', 'TIEMPO_Inmediato'}
support: 0.132, confidence: 0.894, lift: 1.599
-----
{'ESTATUS_DESAPARECIDA', 'TIEMPO_Inmediato', 'EDAD_Joven'} → {'SEXO_HOMBRE'}
support: 0.181, confidence: 0.770, lift: 1.594
-----
{'ESTATUS_DESAPARECIDA', 'EDAD_Joven'} → {'SEXO_HOMBRE'}
support: 0.190, confidence: 0.770, lift: 1.594
-----
{'SEXO_MUJER